# Practice Session 03: Management of networks data

<font size="+2" color="blue">Additional results: recipes</font>

In this session we will study an application of complex networks analysis to cooking. We will start with the *flavors network*, a bi-partite network connecting culinary ingredients to flavour compounds [*].

The initial dataset, prepared by [Ling Cheng in 2016](https://github.com/lingcheng99/Flavor-Network), contains three files:

* `ingredients.tsv` -- information about culinary ingredients
* `compounds.tsv` -- information about flavour compounds
* `ingredient-compound.tsv` -- flavour compounds present in each culinary ingredient
* `recipes.csv` -- ingredients used in recipes around the world (used only for extra points)

[*] Ahn, Y. Y., Ahnert, S. E., Bagrow, J. P., & Barabasi, A. L. (2011). [Flavor network and the principles of food pairing](https://doi.org/10.1038/srep00196). Scientific reports, 1(1), 1-7.


<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="blue">Luca Franceschi</font>

E-mail: <font color="blue">luca.franceschi01@estudiant.upf.edu</font>

Date: <font color="blue">Due Oct. 12th, 18:30</font>

# 1. The flavors bi-partite graph

## 1.0. Examine your input files

Before you begin, we highly recommend you to:

1. Copy the input files to a local directory in your computer 
2. Open them in a spreadsheet and look at them

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 1.1. Read the bipartite graph in a dataframe


The following code, which you can leave as-is, reads the ingredient-compound relationship into a dataframe.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [1]:
# Feel free to add imports if you need them

import io
import csv
import pandas as pd
import networkx as nx

from networkx.algorithms import bipartite

import numpy as np
import matplotlib
import scipy

import itertools

from IPython.display import Image

In [2]:
# Leave this code as-is

INPUT_INGR_FILENAME = "ingredients.tsv"
INPUT_COMP_FILENAME = "compounds.tsv"
INPUT_INGR_COMP_FILENAME = "ingredient-compound.tsv"

In [3]:
# Leave this code as-is

ingredients = pd.read_csv(INPUT_INGR_FILENAME, sep="\t")
display(ingredients.head(3))

compounds = pd.read_csv(INPUT_COMP_FILENAME, sep="\t")
display(compounds.head(3))

ingr_comp = pd.read_csv(INPUT_INGR_COMP_FILENAME, sep="\t")
display(ingr_comp.head(3))


,ingredient_id,ingredient_name,ingredient_category
0,0,magnolia_tripetala,flower
1,1,calyptranthes_parriculata,plant
2,2,chamaecyparis_pisifera_oil,plant derivative


,compound_id,compound_name,compound_code
0,0,jasmone,488-10-8
1,1,5-methylhexanoic_acid,628-46-6
2,2,l-glutamine,56-85-9


,ingredient_id,compound_id
0,1392,906
1,1259,861
2,1079,673


## 1.2. Create the flavors bipartite network


Create a new dataframe named `flavors` by joining `ingredients` and `compounds`.

*Tips*:

* To [join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) a DataFrame A and a DataFrame B using a column X, use `result = A.set_index('X').join(B.set_index('X'), how='inner')
* You will need to do two joins to solve this. First, join `ingredients` and `ingr_comp`, then join the result with `compounds`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [4]:
# Create the `flavors` dataframe and show its first 20 rows

flavors = ingredients.set_index('ingredient_id').join(ingr_comp.set_index('ingredient_id'), how='inner')
flavors = flavors.set_index('compound_id').join(compounds.set_index('compound_id'), how='inner')
flavors = flavors.reset_index() # headers did not seem right
display(flavors.head(20))

,compound_id,ingredient_name,ingredient_category,compound_name,compound_code
0,0,red_bean,vegetable,jasmone,488-10-8
1,0,jasmine_tea,plant derivative,jasmone,488-10-8
2,0,jasmine,flower,jasmone,488-10-8
3,0,soybean,vegetable,jasmone,488-10-8
4,0,dried_black_tea,plant derivative,jasmone,488-10-8
5,0,ceylon_tea,plant derivative,jasmone,488-10-8
6,0,pittosporum_glabratum,plant,jasmone,488-10-8
7,0,mung_bean,vegetable,jasmone,488-10-8
8,0,fermented_tea,plant derivative,jasmone,488-10-8
9,0,fermented_russian_black_tea,plant derivative,jasmone,488-10-8



Drop the code of the compound from the resulting dataframe, sort by ingredient then by compound, and reset its index.

*Tips:*

* To [drop column](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) x from DataFrame A, you can do: `A = A.drop(columns=['x'])`
* To [sort](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) a DataFrame A by column *x*, then by column *y*, you can do: `A = A.sort_values(['x', 'y'])`
* To [reset the index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) of a DataFrame A, you can do: `A = A.reset_index(drop=True)`; the index is the column appearing in boldface in front of every row of a DataFrame

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [5]:
# Modify the `flavors` dataframe as explained above, and show its first 20 rows

flavors = flavors.drop(columns=['compound_code'])
flavors = flavors.sort_values(['ingredient_name', 'compound_id'])
flavors = flavors.reset_index(drop=True)
display(flavors.head(20))

,compound_id,ingredient_name,ingredient_category,compound_name
0,906,abies_alba,plant,bornyl_acetate
1,861,abies_alba_pine_needle,plant,maltol
2,673,abies_balsamea_oil,plant derivative,myrcene
3,906,abies_canadensis,plant,bornyl_acetate
4,906,abies_concolor,plant,bornyl_acetate
5,171,abies_sibirica,plant,camphene
6,278,abies_sibirica,plant,isoborneol
7,906,abies_sibirica,plant,bornyl_acetate
8,165,acacia,plant,eugenol
9,171,acacia,plant,camphene


Write this dataframe to a `flavors.tsv` file, which should be a tab-separated file containing the three fields `ingredient_name`, `ingredient_category` and `compound_name`. Use the function [pandas.DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [6]:
# Save *flavors* into a tab-separated file

flavors.to_csv('flavors.tsv', '\t', columns=['ingredient_name', 'ingredient_category', 'compound_name'], index=False)

## 1.3. Open this bi-partite network in Cytoscape


### 1.3.1. Examine the file you generated

Open the ``flavors.tsv`` file in a spreadsheet program to make sure you generated it correctly; it should have exactly 3 tab-separated columns.

### 1.3.2. Import this file in Cytoscape

Remember these files are imported with ``File > Import > Network from File ...``. Then, you have to select:

* ingredient_name as a ``Source Node``
* ingredient_category as a ``Source Node Attribute``.
* compound_name as a ``Target Node`` 

### 1.3.3. Draw a small part of this graph

Find the `garlic` node and everything connected to it at distance 1 or 2. To do this, find "garlic" and then click on the "two-houses" (neighbors) icon twice. Extract the selected nodes as a sub-graph by doing `File > New network > From selected nodes, all edges`.

Run the network analyzer and then perform `Layout > Edge weighted spring embedded layout` using edge betweenness.

Style the network so that ingredient nodes have a color that depends on their category, using any color except black, and setting black to be the default node color so that compound nodes remain in color white. Set the label color to black. Set the node shape to ellipse. 

Save the image as `flavors.png`; the next cell should display it. This time, node labels do not need to be visible or readable, we just want to appreciate overall clusters.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [7]:
# KEEP THIS CELL AS-IS

# Just adjust width/height if necessary

Image(url="flavors.png", width=1200)

### 1.3.4 Compounds in common

*Onion* and *Garlic* get their distinctive smell from sulfur-containing compounds. How many compounds onion and garlic have in common? Based solely on their names, how many of them do you think contain sulfur?

To answer this question, extract the nodes *Onion*, *Garlic*, and all their compounds in common as a graph. Layout is a hierarchical graph and modify it so that *Onion* appears at the top of the image, the compounds in the middle, and *Garlic* at the bottom of the image.

Save the image as `compounds-in-common.png`; the next cell should display it.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [8]:
# KEEP THIS CELL AS-IS

# Just adjust width/height if necessary

Image(url="compounds-in-common.png", width=1200)

<font size="+1" color="red">Replace this cell by a brief commentary indicating how many compounds they have in common, how many of them seem to contain sulfur (based on their names). Name a couple of those sulfur-containing compounds.</font>

We can see that Garlic and Onion have in common 21 compounds, of which 11 seem to contain sulfur. A couple of those compounds are: propyl-disulfide, methyl_propyl_disulfide or allyl_methyl_trisulfide.

# 2. The ingredient-ingredient graph

The bi-partite flavors graph is hard to visualize as it mixes ingredients and compounds. We will now try to visualize only the connections between ingredients.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>


## 2.1. Create an ingredient-ingredient.csv file


First, copy the list of ingredient names into an array `ingredients_array`. To convert column *x* of DataFrame *A* to an array, use `np.asarray(A['x'])`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [9]:
# Create `ingredients_array` with the list of ingredients and to print the number of ingredients

ingredients_array = np.asarray(ingredients['ingredient_name'])
print("There are %d ingredients" % (len(ingredients_array)))

There are 1530 ingredients


Then, create a dictionary named `ingredient_to_compounds`, in which keys are ingredients, and values are sets of compounds. To create an empty set, you can use `s = set()`. To add to a set, you can do `s.add(element)`. Your code should look like this:

```python
ingredients_array = ...
print("There are %d ingredients" % (len(ingredients_array)))

ingredient_to_compounds = {}

for index, row in flavors.iterrows():
    ...

```

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [10]:
# Create dictionary `ingredient_to_compounds` with a set of compounds for each ingredient. Print the number of 
# keys of this dictionary. It should be less than or equal to the number of ingredients

ingredient_to_compounds = {}

for index, row in flavors.iterrows():
	if row['ingredient_name'] not in ingredient_to_compounds:
		ingredient_to_compounds[row['ingredient_name']] = set()
	ingredient_to_compounds[row['ingredient_name']].add(row['compound_name'])

print('There are %d items in the dictionary' % len(ingredient_to_compounds))

There are 1525 items in the dictionary


Next, we create a NetworkX graph with nodes representing ingredients and edges of weight `x` connecting two ingredients having `x` flavor compounds in common.

To create an empty graph, do `ingredient_ingredient = nx.Graph()`.

Now, iterate through all pairs of ingredients in `ingredients_array` and compute the compounds they have in common between them. To iterate through all pair combinations of an array X, you can use:

```
for u, v in itertools.combinations(X,2):
    ...

```

The size of the intersection of two lists of compounds `l1`, `l2` can be obtained with `len(l1.intersection(l2))`. This will be the weight of the edge connecting two ingredients corresponding to those lists.

Please note you may need to check whether both ingredients have compounds. You can test it by asking `if u in ingredient_to_compounds and v in ingredient_to_compounds`

To facilitate visualization, we will keep only edges connecting two ingredients having **MIN_COMMON_COMPOUNDS or more compounds in common**. Set the value of **MIN_COMMON_COMPOUNDS** so that the resulting graph has somewhere around 150 +/- 30 nodes.

To add to graph *G* an edge between nodes *u* and *v* having weight *w*, do `G.add_edge(u, v, weight=w)`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [11]:
# Create the `ingredient_ingredient` graph
MIN_COMMON_COMPOUNDS = 70

ingredient_ingredient = nx.Graph()

for u, v in itertools.combinations(ingredients_array, 2):
	if u in ingredient_to_compounds and v in ingredient_to_compounds:
		weight = len(ingredient_to_compounds[u].intersection(ingredient_to_compounds[v]))
		if weight >= MIN_COMMON_COMPOUNDS:
			ingredient_ingredient.add_node(u)
			ingredient_ingredient.add_node(v)
			ingredient_ingredient.add_edge(u, v, weight=weight)

In [12]:
# Leave as-is
print("The ingredient-ingredient graph has %d nodes and %d edges" %
      (ingredient_ingredient.number_of_nodes(), ingredient_ingredient.number_of_edges()))

The ingredient-ingredient graph has 165 nodes and 2153 edges


Save the resulting graph into a file. You can use [write_gml](https://networkx.org/documentation/stable/reference/readwrite/generated/networkx.readwrite.gml.write_gml.html#networkx.readwrite.gml.write_gml) to use the *GML* format.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [13]:
OUTPUT_INGR_INGR_FILENAME = 'ingredient-ingredient.gml'

In [14]:
# Save graph G to file OUTPUT_INGR_INGR_FILENAME

nx.write_gml(ingredient_ingredient, OUTPUT_INGR_INGR_FILENAME)

## 2.2. Work with this file in Cytoscape

## 2.2.1. Inspect this file

*Tip:* Open the ``ingredient-ingredient.gml`` file in a text editor first to see how it is structured.


## 2.2.2. Import this file into Cytoscape

To import this file into Cytoscape:

* `File > Import > Network from file ...`
* Open the `ingredient-ingredient.gml` file

Now we need to import ingredient categories:

* `File > Import > Table from file ...`
* Open the `ingredients.tsv` file
* Import data as "Node Table Columns"
* `ingredient_name`: key
* `ingredient_category`: attribute

Do a `Layout > Edge weighted spring embedded` layout on the *weight* attribute.

### 2.2.3. Style and add simple annotations

Style lines connecting nodes so their thickness and color reflects the number of compounds in common.

Color the nodes with colors representing the ingredient categories. Note that if you right-click on "Mapping type" when creating a discrete mapping, you can use an automatic mapping generator to start with.

Save the main connected component of this graph as `ingr-ingr.png` using `File > Export > Network to image ...`.

Save a legend as `ingr-ingr-legend.gif` using the hamburger menu in `Style`and selecting `Create legend ...`

The next cell should display your graph and its legend.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [15]:
# Change width if necessary

display(Image(url="ingr-ingr.png", width=1200))

display(Image(url="ingr-ingr-legend.gif", width=400))

<font size="+1" color="red">Replace this cell by two interested pairings (combinations of two or more items) suggested by this network. By a pairing we mean food items that may taste good together, and a plausible explanation considering the network structure.</font>

Pairing 1: Coffee and peanut butter. We could see this pairing very often in a breakfast. We can see that these two ingredients have in common 83 compounds. Both are plant derivatives.

Pairing 2: Grilled beef and french fried potato. It is a very well-known pairing (steak fries) used in many cuisines with its origin in Belgium. We can see that these two ingredients have in common 86 compounds. In this case the french fries are a vegetable and grilled beef is meat.

It seems feasible that if two ingredients share a reasonable amount of compounds, they will have similar tastes or will be able to 'match' very well. Especially if the two ingredients are in a different category, such as pairing number two.

## Extra section

In [60]:
import random

def string_to_ingredients(row):
	raw_list = row.split(',')
	raw_list.pop(0)
	return raw_list

def extract_random(file_csv, amount=3):
	random.seed() # Set seed
	dictionary = {}
	for i in range(amount):
		dictionary[i] = set()
		row = file_csv.iloc[random.randint(4, file_csv.index[-1])].to_string() # Take random row
		dictionary[i].update(string_to_ingredients(row))
	return dictionary
		

def save_gml(recipes):
	for i in range(len(recipes)):
		nx.write_gml()

In [61]:
recipes_csv = pd.read_csv('recipes.csv', sep='\t') # Import the recipes.csv as plain text, since we cannot format it directly
recipes_csv.reset_index()
# display(recipes_csv)
recipes = extract_random(recipes_csv) # Extracted recipes maps the number of recipe to the ingredients it has

save_gml(recipes)

{0: {'c...', 'vinegar', 'sesame_oil', 'lemon', 'soy_sauce'}, 1: {'lemon_juice', 'vegeta...', 'cane_molasses'}, 2: {'wheat', 'butter', 'coconut', 'lemon_pe...'}}


# DELIVER (individually)

Read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/networks-science-course/blob/master/upf/upf-evaluation.md).

Deliver a zip file containing:

* This notebook
* The ``flavors.tsv``, ``flavors.png``, and ``flavors-legend.gif`` files
* The ``ingredient-ingredient.tsv``, ``ingr-ingr.png``, and ``ingr-ingr-legend.gif`` files

## Extra points available

For more learning and extra points, get the `recipes.csv` file. It contains one recipe per line, in this format:

```
EastAsian,roasted_sesame_seed,garlic,cayenne,seaweed,sesame_oil
```

This means there is one East Asian dish whose recipe requires the ingredients "roasted_sesame_seed", "garlic", "cayenne", "seaweed", and "sesame_oil".

Select 3 recipes and draw using Cytoscape a graph with their ingredients and the compounds in those ingredients. Include those subgraphs here, plus a brief commentary about whether the ingredients used share many compounds, few compounds, or not at all, and any other observations you want to make about the selected recipes.

**Note:** if you go for the extra points, add ``<font size="+2" color="blue">Additional results: recipes</font>`` at the top of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>
